# mT5-small Fine-tuning for English-to-Urdu Translation

Lightweight multilingual model for machine translation.

**Model:** google/mt5-small (~300M parameters - fits in 14GB GPU)

**Dataset:** Parallel Corpus for English-Urdu Language (24,525 sentence pairs)

## 1. Imports and Setup

In [ ]:
import os
import re
import gc
import random
import torch
import math
from torch.utils.data import Dataset
from collections import Counter

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Data Loading and Preprocessing

In [ ]:
def load_data():
    en_path = "Dataset/english-corpus.txt"
    ur_path = "Dataset/urdu-corpus.txt"

    with open(en_path, "r", encoding="utf-8") as f:
        en_lines = [line.strip() for line in f.readlines()]
    with open(ur_path, "r", encoding="utf-8") as f:
        ur_lines = [line.strip() for line in f.readlines()]

    assert len(en_lines) == len(ur_lines), "Mismatch in number of lines"
    print(f"Loaded {len(en_lines)} sentence pairs")
    return en_lines, ur_lines


def clean_english(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-zA-Z0-9\s.,!?'\-]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_urdu(text):
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def preprocess_data(en_lines, ur_lines):
    pairs = []
    for en, ur in zip(en_lines, ur_lines):
        en_clean = clean_english(en)
        ur_clean = clean_urdu(ur)
        if len(en_clean) > 0 and len(ur_clean) > 0:
            pairs.append((en_clean, ur_clean))
    print(f"After cleaning: {len(pairs)} pairs")
    return pairs


en_lines, ur_lines = load_data()
pairs = preprocess_data(en_lines, ur_lines)

# Split data
random.seed(42)
random.shuffle(pairs)
n = len(pairs)
n_train = int(n * 0.9)
n_val = int(n * 0.05)

train_pairs = pairs[:n_train]
val_pairs = pairs[n_train:n_train + n_val]
test_pairs = pairs[n_train + n_val:]

print(f"\nTrain: {len(train_pairs)}, Val: {len(val_pairs)}, Test: {len(test_pairs)}")
print(f"\nSample pairs:")
for i in range(3):
    print(f"  EN: {train_pairs[i][0]}")
    print(f"  UR: {train_pairs[i][1]}")
    print()

## 3. Load mT5-small Model

In [ ]:
# === Use mT5-small (much smaller than mBART-large-50) ===
model_name = "google/mt5-small"  # ~300M params vs mBART's 600M+
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = model.cuda()

# Enable gradient checkpointing
model.gradient_checkpointing_enable()

print("✓ mT5-small model loaded with gradient checkpointing")
print(f"  Model size: ~300M parameters (vs mBART ~600M+)")
print(f"  This will fit comfortably in 14GB GPU")

## 4. Dataset Class

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len=32):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        en, ur = self.pairs[idx]
        
        # Prefix for mT5 (source language hint)
        en_prefixed = f"translate English to Urdu: {en}"
        
        model_inputs = self.tokenizer(
            en_prefixed,
            max_length=self.max_len, 
            truncation=True, 
            padding="max_length"
        )
        
        labels = self.tokenizer(
            ur,
            max_length=self.max_len, 
            truncation=True, 
            padding="max_length"
        )

        return {
            "input_ids": torch.tensor(model_inputs["input_ids"]).squeeze(),
            "attention_mask": torch.tensor(model_inputs["attention_mask"]).squeeze(),
            "labels": torch.tensor(labels["input_ids"]).squeeze(),
        }


# Use subset of data
print(f"Full train set: {len(train_pairs)} samples")
mt5_train = TranslationDataset(train_pairs[:3000], tokenizer)
mt5_val = TranslationDataset(val_pairs[:500], tokenizer)

print(f"Train samples used: {len(mt5_train)}")
print(f"Val samples used: {len(mt5_val)}")

## 5. Training Configuration

In [ ]:
# === MEMORY-EFFICIENT SETTINGS FOR SMALL MODEL ===
training_args = Seq2SeqTrainingArguments(
    output_dir="mt5_checkpoints",
    num_train_epochs=1,
    
    per_device_train_batch_size=2,   # Can use batch size 2 with smaller model
    gradient_accumulation_steps=1,
    
    per_device_eval_batch_size=2,
    eval_strategy="no",
    save_strategy="no",
    
    learning_rate=1e-4,
    weight_decay=0.01,
    predict_with_generate=False,
    
    logging_steps=50,
    report_to="none",
    seed=42,
    remove_unused_columns=False,
    max_steps=2000,  # Increase training duration (3000 samples / batch_size 2 = ~1500 steps per epoch, so 2000 = ~1.3 epochs)
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=mt5_train,
    eval_dataset=None,
    data_collator=data_collator,
)

print("✓ Trainer configured with mT5-small")
print("  - Model: google/mt5-small (~300M params)")
print("  - Batch size: 2")
print("  - Max length: 32")
print("  - Training samples: 3000")
print("  - Max steps: 2000  (Increase this to train longer)")

## 6. Train the Model

In [ ]:
print("\n" + "="*70)
print("Starting mT5-small Fine-tuning...")
print("="*70 + "\n")

trainer.train()

model.save_pretrained("mt5_finetuned")
tokenizer.save_pretrained("mt5_finetuned")
print("\n✓ Training complete! Model saved to 'mt5_finetuned/'")

# Clear GPU memory
gc.collect()
torch.cuda.empty_cache()
print("✓ GPU memory cleared")

## 7. Inference Function

In [ ]:
def translate_mt5(text, model, tokenizer):
    """Translate English text to Urdu using fine-tuned mT5."""
    input_text = f"translate English to Urdu: {text}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=32, truncation=True).to(model.device)
    
    # Better generation parameters
    generated = model.generate(
        **inputs,
        max_length=32,
        num_beams=4,           # Beam search for better quality
        early_stopping=True,
        no_repeat_ngram_size=2,  # Prevent repetition
        temperature=0.7,
        top_p=0.9,
        do_sample=False,
    )
    
    result = tokenizer.decode(generated[0], skip_special_tokens=True)
    
    # Clean up any remaining sentinel tokens
    import re
    result = re.sub(r'<extra_id_\d+>', '', result).strip()
    
    return result

print("✓ Inference function improved")

## 8. Test Translations

In [ ]:
test_sentences = [
    "how are you",
    "what is your name",
    "i love my country",
    "the weather is beautiful today",
    "where is the school",
    "he is a good person",
    "please help me",
    "i am going home",
]

print("\n" + "="*70)
print("mT5-small English-to-Urdu Translations (Fine-tuned)")
print("="*70)
for sent in test_sentences:
    translation = translate_mt5(sent, model, tokenizer)
    print(f"EN: {sent}")
    print(f"UR: {translation}")
    print("-" * 70)

## 9. Evaluate on Test Set

In [ ]:
def compute_bleu(reference, hypothesis, max_n=4):
    """Compute BLEU score for a single reference-hypothesis pair."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if len(hyp_tokens) == 0:
        return 0.0

    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter()
        for i in range(len(ref_tokens) - n + 1):
            ngram = tuple(ref_tokens[i:i+n])
            ref_ngrams[ngram] += 1

        hyp_ngrams = Counter()
        for i in range(len(hyp_tokens) - n + 1):
            ngram = tuple(hyp_tokens[i:i+n])
            hyp_ngrams[ngram] += 1

        clipped = sum(min(hyp_ngrams[ng], ref_ngrams[ng]) for ng in hyp_ngrams)
        total = max(sum(hyp_ngrams.values()), 1)
        precisions.append(clipped / total)

    if any(p == 0 for p in precisions):
        return 0.0

    log_avg = sum(math.log(p) for p in precisions) / max_n
    bp = 1.0
    if len(hyp_tokens) < len(ref_tokens):
        bp = math.exp(1 - len(ref_tokens) / max(len(hyp_tokens), 1))

    return bp * math.exp(log_avg)


def corpus_bleu(references, hypotheses):
    """Compute average BLEU score over a corpus."""
    scores = [compute_bleu(ref, hyp) for ref, hyp in zip(references, hypotheses)]
    return sum(scores) / max(len(scores), 1)


# Evaluate on first 100 test samples
print("\nEvaluating on test set...")
references, hypotheses = [], []
for en_s, ur_s in test_pairs[:100]:
    pred = translate_mt5(en_s, model, tokenizer)
    references.append(ur_s)
    hypotheses.append(pred)

test_bleu = corpus_bleu(references, hypotheses)
print(f"\nTest BLEU Score (first 100 samples): {test_bleu:.4f}")
print(f"Total test samples: {len(test_pairs)}")

print("\n" + "="*80)
print(f"{'English':<35} | {'Predicted Urdu':<35}")
print("="*80)
for i in range(min(10, len(test_pairs))):
    en_s = test_pairs[i][0]
    ref = test_pairs[i][1]
    hyp = hypotheses[i]
    print(f"EN:   {en_s}")
    print(f"REF:  {ref}")
    print(f"PRED: {hyp}")
    print("-"*80)